# Formula 1 podium prediction: data cleaning

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "f1_clean.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.f1_clean import (
    ASSIGNMENT_LAST_YEAR,
    SENTINEL,
    clean_tables,
    collapse_entries,
    load_raw,
    project_paths,
    raw_hashes,
    replace_sentinels,
)

PATHS = project_paths(ROOT)
PATHS["figures"].mkdir(parents=True, exist_ok=True)
HASHES_AT_START = raw_hashes(PATHS["raw"])

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 120)

NAVY = "#1f4e79"
CLAY = "#c46b2c"
TEAL = "#2f6f6a"


def styled(ax, title, xlabel, ylabel, grid="y"):
    ax.set_title(title, loc="left", fontsize=13, pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid, color="#e6e6e6")
    ax.set_axisbelow(True)


def save(fig, name):
    fig.tight_layout()
    fig.savefig(PATHS["figures"] / name, dpi=140, bbox_inches="tight", facecolor="white")
    plt.show()


raw = load_raw(PATHS["raw"])
print(f"Loaded {len(raw)} tables from {PATHS['raw']}")

## 2. What is in the files

In [ ]:
inventory = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in raw.items()
])
display(inventory)

races_raw = replace_sentinels(raw["races"]).copy()
races_raw["year"] = pd.to_numeric(races_raw["year"])
print(
    f"Seasons in races: {int(races_raw['year'].min())}–{int(races_raw['year'].max())}. "
    f"Distinct seasons: {races_raw['year'].nunique()}."
)
print(f"Assignment window named in the brief: 1950–{ASSIGNMENT_LAST_YEAR}.")

In [ ]:
results_year = replace_sentinels(raw["results"])[["raceId"]].merge(
    races_raw[["raceId", "year"]], on="raceId", how="left"
)
volume = pd.DataFrame({
    "races": races_raw.groupby("year").size(),
    "result_rows": results_year.groupby("year").size(),
}).fillna(0)

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(volume.index, volume["races"], color=TEAL, linewidth=2, label="Races")
ax.plot(volume.index, volume["result_rows"], color=NAVY, linewidth=2, label="Result rows")
styled(ax, "Race calendar and result rows by season", "Season", "Count")
ax.legend(frameon=False)
save(fig, "01_season_volume.png")

## 3. Sentinel values, before they are converted

In [ ]:
from src.f1_clean import summarize_sentinels

sentinels = summarize_sentinels(raw)
display(sentinels)

status_counts = raw["constructor_results"]["status"].value_counts()
print("constructor_results.status raw values:")
display(status_counts.to_frame("rows"))

plot_df = sentinels.sort_values("sentinel_share")
fig, ax = plt.subplots(figsize=(10, max(6, 0.28 * len(plot_df))))
ax.barh(
    plot_df["table"] + "." + plot_df["column"],
    plot_df["sentinel_share"] * 100,
    color=NAVY,
)
styled(
    ax,
    "Share of cells holding the \\N sentinel",
    "Percent of rows",
    "",
    grid="x",
)
save(fig, "02_sentinel_share.png")

In [ ]:
results_status = replace_sentinels(raw["results"]).merge(
    replace_sentinels(raw["status"]), on="statusId", how="left"
)
top_status = results_status["status"].value_counts().head(12).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_status.index, top_status.values, color=NAVY)
styled(ax, "Twelve most common finishing statuses", "Result rows", "", grid="x")
save(fig, "03_status_counts.png")
print(f"Distinct statuses used in results: {results_status['status'].nunique()}")

## 4. Keys before cleaning

In [ ]:
from src.f1_clean import audit_keys

keys_before = audit_keys(raw)
failed_before = keys_before.loc[~keys_before["passed"]].copy()
print(f"{int(keys_before['passed'].sum())} of {len(keys_before)} checks passed before cleaning.")
display(failed_before)

## 5. Repeated driver-race rows

In [ ]:
results_view = replace_sentinels(raw["results"]).copy()
for column in ["grid", "positionOrder", "laps", "points"]:
    results_view[column] = pd.to_numeric(results_view[column])

drivers_view = replace_sentinels(raw["drivers"]).copy()
drivers_view["driver"] = drivers_view["forename"] + " " + drivers_view["surname"]

repeated = results_view.loc[results_view.duplicated(["raceId", "driverId"], keep=False)].copy()
repeated = repeated.merge(races_raw[["raceId", "year", "name"]], on="raceId", how="left")
repeated = repeated.merge(drivers_view[["driverId", "driver"]], on="driverId", how="left")
pairs = repeated.drop_duplicates(["raceId", "driverId"])

print(f"Repeated result rows: {len(repeated)}")
print(f"Driver-race pairs involved: {len(pairs)}")
print(f"Seasons involved: {int(pairs['year'].min())}–{int(pairs['year'].max())}")

year_counts = pairs.groupby("year").size().reindex(range(1950, 1979), fill_value=0)
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.bar(year_counts.index, year_counts.values, color=NAVY, width=0.8)
styled(ax, "Driver-race pairs that appear more than once", "Season", "Pairs")
save(fig, "04_duplicate_pairs_by_year.png")

In [ ]:
french_id = races_raw.loc[
    (races_raw["year"] == 1951) & (races_raw["name"] == "French Grand Prix"),
    "raceId",
].iloc[0]
french = repeated.loc[repeated["raceId"] == french_id].copy()
shared_numbers = french.groupby("number")["driverId"].nunique()
shared_numbers = shared_numbers[shared_numbers > 1].index
french_shared = french.loc[french["number"].isin(shared_numbers), [
    "driver", "number", "grid", "positionText", "positionOrder", "laps", "points",
]].sort_values(["number", "positionOrder", "driver"])
display(french_shared)

In [ ]:
late = repeated.loc[repeated["year"] >= 1965, [
    "year", "name", "driver", "constructorId", "grid", "positionText", "positionOrder", "laps",
]].sort_values(["year", "driver", "positionOrder"])
print(f"Repeated rows after 1964: {len(late)}")
display(late)

status_view = replace_sentinels(raw["status"])
late_status = repeated.loc[repeated["year"] >= 1965, ["statusId"]].merge(status_view, on="statusId")
print("Statuses on those rows:")
display(late_status["status"].value_counts().to_frame("rows"))

## 6. What “podium” means in this file

In [ ]:
results_view["brief_podium"] = results_view["positionOrder"] <= 3
results_view["official_podium"] = results_view["positionText"].isin(["1", "2", "3"])
brief_only = results_view["brief_podium"] & ~results_view["official_podium"]
official_only = results_view["official_podium"] & ~results_view["brief_podium"]
print(f"Brief podium rows: {int(results_view['brief_podium'].sum())}")
print(f"Official podium rows: {int(results_view['official_podium'].sum())}")
print(f"In the brief label only: {int(brief_only.sum())}")
print(f"In the official label only: {int(official_only.sum())}")

conflict_ids = results_view.loc[brief_only, "raceId"].unique()
conflict_race = results_view.loc[results_view["raceId"].isin(conflict_ids)].copy()
conflict_race = conflict_race.merge(drivers_view[["driverId", "driver"]], on="driverId", how="left")
conflict_race = conflict_race.merge(races_raw[["raceId", "year", "name"]], on="raceId", how="left")
conflict_race = conflict_race.merge(status_view, on="statusId", how="left")
show_cols = ["year", "name", "driver", "grid", "positionText", "positionOrder", "points", "status"]
display(
    conflict_race.sort_values(["raceId", "positionOrder"])[show_cols]
    .head(8)
)

## 7. Coverage the later feature work has to respect

In [ ]:
result_race_ids = set(results_view["raceId"])
unraced = races_raw.loc[~races_raw["raceId"].isin(result_race_ids), ["year", "round", "name", "date"]]
unraced = unraced.sort_values(["year", "round"])
print(f"Scheduled races with no result rows: {len(unraced)}")
display(unraced)

quali_ids = set(replace_sentinels(raw["qualifying"])["raceId"])
completed = races_raw.loc[races_raw["raceId"].isin(result_race_ids)].copy()
completed["has_qualifying"] = completed["raceId"].isin(quali_ids)
coverage = completed.groupby("year")["has_qualifying"].mean()
print(f"First season present in qualifying: {int(coverage[coverage > 0].index.min())}")

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(coverage.index, coverage.values * 100, color=NAVY, linewidth=2)
ax.axvspan(1994, 2002.5, color="#f3e3d3", zorder=0)
styled(ax, "Completed races that have a qualifying table", "Season", "Percent of races")
ax.set_ylim(-5, 105)
save(fig, "05_qualifying_coverage.png")

In [ ]:
grid_null = results_view.loc[results_view["grid"].isna(), ["raceId"]].merge(
    races_raw[["raceId", "year", "name", "date"]], on="raceId"
)
print("Rows with a null grid, by race:")
display(grid_null.groupby(["year", "name", "date"]).size().rename("null_grid_rows").reset_index())
print(f"Rows with grid 0: {int((results_view['grid'] == 0).sum())}")

debut = results_view.merge(races_raw[["raceId", "year"]], on="raceId").groupby("driverId")["year"].min()
driver_codes = drivers_view.merge(debut.rename("debut"), left_on="driverId", right_index=True, how="inner")
driver_codes["decade"] = (driver_codes["debut"] // 10 * 10).astype(int)
driver_codes["code_missing"] = driver_codes["code"].isna()
driver_codes["number_missing"] = driver_codes["number"].isna()
code_by_decade = driver_codes.groupby("decade").agg(
    drivers=("driverId", "size"),
    code_missing=("code_missing", "mean"),
    number_missing=("number_missing", "mean"),
)
code_by_decade[["code_missing", "number_missing"]] = code_by_decade[["code_missing", "number_missing"]] * 100
display(code_by_decade.round(1))

fig, ax = plt.subplots(figsize=(10, 4.4))
x = np.arange(len(code_by_decade))
width = 0.38
ax.bar(x - width / 2, code_by_decade["code_missing"], width=width, color=NAVY, label="3-letter code missing")
ax.bar(x + width / 2, code_by_decade["number_missing"], width=width, color=CLAY, label="Permanent number missing")
styled(ax, "Missing modern driver identifiers by decade of debut", "Decade of first result", "Percent of drivers")
ax.set_xticks(x, [str(int(year)) + "s" for year in code_by_decade.index], rotation=0)
ax.set_ylim(0, 105)
ax.legend(frameon=False)
save(fig, "06_driver_identifier_missingness.png")

In [ ]:
standings = replace_sentinels(raw["driver_standings"]).copy()
standings["points"] = pd.to_numeric(standings["points"])
unranked = standings.loc[standings["position"].isna()].merge(
    races_raw[["raceId", "year", "name"]], on="raceId"
)
print(f"Driver-standings rows with no position: {len(unranked)}")
display(
    unranked.groupby(["year", "name", "positionText"])
    .size()
    .rename("rows")
    .reset_index()
)

## 8. The cleaning rules

In [ ]:
result = clean_tables(raw)

print("Coercion failures (empty means every present value cast successfully):")
display(result.coercion_failures)

print("Parsed strings against the millisecond columns already in the file:")
display(result.time_checks)

print(f"Disqualified constructor results: {int(result.cleaned['constructor_results']['disqualified'].sum())}")

print("Missing-value method for every column that had a blank marker:")
display(result.missing_policy)
left_blank = result.missing_policy.loc[result.missing_policy["method"] == "leave missing", "values_filled"]
assert left_blank.eq(0).all()

In [ ]:
q1 = result.cleaned["qualifying"][["raceId", "driverId", "q1", "q1_seconds"]].dropna(subset=["q1_seconds"])
ordinary = q1.loc[q1["q1_seconds"] <= 180, "q1_seconds"]
odd = q1.loc[q1["q1_seconds"] > 180].merge(
    result.cleaned["races"][["raceId", "year", "name"]], on="raceId"
).merge(
    result.cleaned["drivers"][["driverId", "forename", "surname"]], on="driverId"
)
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.hist(ordinary, bins=40, color=NAVY)
styled(
    ax,
    f"Parsed Q1 times, excluding {len(odd)} value above 3 minutes. Median {ordinary.median():.1f}s",
    "Q1 (seconds)",
    "Qualifying rows",
)
save(fig, "07_q1_seconds.png")
print("Qualifying times longer than 3 minutes, left as parsed from the source string:")
display(odd[["year", "name", "forename", "surname", "q1", "q1_seconds"]])

sample_times = result.cleaned["results"].loc[
    result.cleaned["results"]["winner_time_seconds"].notna()
    | result.cleaned["results"]["gap_seconds"].notna(),
    ["time", "winner_time_seconds", "gap_seconds", "elapsed_seconds"],
].head(6)
display(sample_times)

In [ ]:
keys_after = result.keys_after
failed_after = keys_after.loc[~keys_after["passed"]].copy()
print(f"{int(keys_after['passed'].sum())} of {len(keys_after)} checks passed after cleaning.")
display(failed_after)
print("Lap-time key repair:")
display(result.lap_conflict_summary)

## 9. Which row is kept

In [ ]:
comparison = result.rule_comparison
print(f"Repeated pairs compared: {len(comparison)}")
print(f"Same resultId: {int(comparison['same_row'].sum())}")
print(f"Same positionOrder: {int(comparison['same_position'].sum())}")
print(f"Same podium label: {int(comparison['same_podium'].sum())}")
print(f"Disagreements: {int((~comparison['same_row']).sum())}")

toy = pd.DataFrame([
    {"raceId": 1, "driverId": 7, "resultId": 10, "positionOrder": 8, "laps": 70},
    {"raceId": 1, "driverId": 7, "resultId": 11, "positionOrder": 2, "laps": 12},
])
toy_compare = pd.DataFrame({
    "rule": ["best_classification", "most_laps"],
    "kept_resultId": [
        int(collapse_entries(toy, "best_classification").iloc[0]["resultId"]),
        int(collapse_entries(toy, "most_laps").iloc[0]["resultId"]),
    ],
})
display(toy)
display(toy_compare)

## 10. The driver-race table, and what cleaning changed

In [ ]:
driver_race = result.driver_race
print(f"Result rows: {len(result.cleaned['results'])}")
print(f"Driver-race rows: {len(driver_race)}")
print(f"Rows removed by the collapse: {len(result.cleaned['results']) - len(driver_race)}")
print(f"Unique (raceId, driverId): {not driver_race.duplicated(['raceId', 'driverId']).any()}")
print(f"Shared entries: {int(driver_race['shared_entry'].sum())}")
print(f"Shared entries with more than one constructor: {int((driver_race['n_constructors'] > 1).sum())}")
print(f"Podium definition conflicts: {int(driver_race['podium_definition_conflict'].sum())}")
print(
    f"Rows inside the brief's 1950–{ASSIGNMENT_LAST_YEAR} window: "
    f"{int(driver_race['in_assignment_window'].sum())} of {len(driver_race)}"
)

before = results_view[["raceId", "positionOrder"]].merge(
    races_raw[["raceId", "year"]], on="raceId", how="left"
)
before["podium"] = before["positionOrder"] <= 3
before["decade"] = (before["year"] // 10 * 10).astype(int)
after = driver_race[["raceId", "year", "podium"]].copy()
after["decade"] = (after["year"] // 10 * 10).astype(int)

rate = pd.DataFrame({
    "before": before.groupby("decade")["podium"].mean() * 100,
    "after": after.groupby("decade")["podium"].mean() * 100,
})
field = (
    after.groupby(["decade", "raceId"]).size()
    .groupby("decade").median()
    .rename("median_starters")
)
display(rate.round(2).join(field))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
x = np.arange(len(rate))
width = 0.38
axes[0].bar(x - width / 2, rate["before"], width=width, color="#9bb4c8", label="Before collapse")
axes[0].bar(x + width / 2, rate["after"], width=width, color=NAVY, label="After collapse")
styled(axes[0], "Podium rate by decade", "Decade", "Percent of rows")
axes[0].set_xticks(x, [str(int(year)) + "s" for year in rate.index], rotation=45)
axes[0].legend(frameon=False)

axes[1].bar(x, field.values, color=TEAL)
styled(axes[1], "Median starters per race", "Decade", "Drivers")
axes[1].set_xticks(x, [str(int(year)) + "s" for year in field.index], rotation=45)
save(fig, "08_podium_rate_before_after.png")

In [ ]:
balance = driver_race["podium"].value_counts().rename({False: "Not podium", True: "Podium"})
balance_share = driver_race["podium"].mean()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(balance.index, balance.values, color=[NAVY, CLAY])
styled(ax, f"Class balance on driver_race ({balance_share * 100:.1f}% podium)", "", "Rows")
save(fig, "09_class_balance.png")

multi_podium = driver_race.groupby("raceId")["podium"].sum()
print("Races by how many drivers carry the podium label:")
display(multi_podium.value_counts().sort_index().rename("races").to_frame())

In [ ]:
grid = driver_race.dropna(subset=["grid"]).copy()
grid["grid"] = grid["grid"].astype(int)
grid_rates = grid.groupby("grid")["podium"].agg(rate="mean", starts="size")
plotted = grid_rates.loc[grid_rates["starts"] >= 50].copy()
omitted = int(grid_rates.loc[grid_rates["starts"] < 50, "starts"].sum())

fig, ax = plt.subplots(figsize=(11, 4.4))
colors = [CLAY if grid_pos == 0 else NAVY for grid_pos in plotted.index]
ax.bar(plotted.index.astype(str), plotted["rate"] * 100, color=colors)
styled(ax, "Podium rate by starting grid", "Grid", "Percent of starts")
save(fig, "10_podium_rate_by_grid.png")
grid_zero = driver_race.loc[driver_race["grid"] == 0, "podium"]
print(f"Grid 0 podium rate: {grid_zero.mean():.1%} on {len(grid_zero)} starts")
print(f"Starts on grids with fewer than 50 starts, omitted from the chart: {omitted}")
print(f"Starts with a null grid, also omitted: {int(driver_race['grid'].isna().sum())}")

## 11. Handoff

In [ ]:
from src.f1_clean import DRIVER_RACE_DICTIONARY, save_outputs

dictionary = pd.DataFrame(DRIVER_RACE_DICTIONARY, columns=["column", "role", "meaning"])
display(dictionary)

kept_example = result.duplicate_rows.loc[result.duplicate_rows["raceId"] == int(french_id)].copy()
name_map = (
    drivers_view.assign(driverId=pd.to_numeric(drivers_view["driverId"]))
    .set_index("driverId")["driver"]
)
kept_example["driver"] = kept_example["driverId"].map(name_map)
shared_number_ids = pd.to_numeric(pd.Index(shared_numbers))
display(
    kept_example.loc[kept_example["number"].isin(shared_number_ids), [
        "driver", "number", "grid", "positionText", "positionOrder", "laps", "kept",
    ]].sort_values(["number", "positionOrder", "driver"])
)

assert result.coercion_failures.empty
time_gap = result.time_checks.loc[~(result.time_checks["max_abs_seconds"] < 0.01)]
if time_gap.empty:
    print("Parsed times match the millisecond columns within 0.01 seconds.")
else:
    print("These source columns disagree with the millisecond values by 0.01 seconds or more. Both numbers are kept:")
    display(time_gap)
assert not driver_race.duplicated(["raceId", "driverId"]).any()
assert not result.cleaned["lap_times"].duplicated(["raceId", "driverId", "lap"]).any()
assert HASHES_AT_START == raw_hashes(PATHS["raw"])

save_outputs(result, PATHS)
print("Wrote cleaned tables, driver_race, and audit files.")
print(f"Raw files unchanged: {HASHES_AT_START == raw_hashes(PATHS['raw'])}")